# 05 - Model Baseline: Ekstraksi Fitur Visual dengan ResNet-50
### Proyek CITRA SERUPA — Sistem Rekomendasi Gambar Fashion Berdasarkan Kemiripan Visual

---

**Tujuan Notebook:**
Notebook ini merupakan tahap inti pemodelan deep learning (*Step 06*):
1. **Membangun Model Baseline:** Memanfaatkan arsitektur **ResNet-50** pretrained ImageNet dari PyTorch.
2. **Feature Extractor:** Memotong lapisan klasifikasi akhir (`fc layer`), sehingga model berfungsi sebagai ekstraktor fitur visual yang menghasilkan **vektor representasi 2048 dimensi** untuk setiap gambar.
3. **Normalisasi Vektor (L2 Normalization):** Menormalisasi seluruh vektor fitur agar perhitungan *Cosine Similarity* dapat dilakukan secara instan dengan perkalian matriks (*dot product*).
4. **Penyimpanan Fitur:** Menyimpan matriks fitur ke Google Drive (`CITRA-SERUPA/features/`).
5. **Uji Coba Pencarian Rekomendasi (Top-5 Retrieval):** Mendemonstrasikan sistem pencarian gambar serupa pertama kita (memasukkan 1 gambar kueri dan menemukan 5 gambar paling mirip dari galeri).

## 1. Hubungkan Google Colab ke Google Drive & Persiapan Direktori

In [ ]:
from google.colab import drive
import os
import glob
import torch
import pandas as pd

# 1. Hubungkan Google Drive
drive.mount('/content/drive')

# 2. Deteksi lokasi folder proyek
base_dir = '/content/drive/MyDrive/CITRA-SERUPA'
if not os.path.exists(base_dir):
    base_dir = '/content/drive/MyDrive/CITRA_SERUPA'

# Cari folder dataset dan images yang tepat
dataset_dir = None
images_dir = None
for d in ['dataset', 'dataset ']:
    cand = os.path.join(base_dir, d, 'images')
    if os.path.exists(cand) and len(os.listdir(cand)) > 0:
        dataset_dir = os.path.join(base_dir, d)
        images_dir = cand
        break

csv_path = os.path.join(dataset_dir, 'metadata.csv')
features_dir = os.path.join(base_dir, 'features')
os.makedirs(features_dir, exist_ok=True)

# Cek Device (GPU T4)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=" * 65)
print(f"⚡ Akselerator Komputasi : {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"📁 Folder Gambar        : {images_dir}")
print(f"📄 File Metadata        : {csv_path}")
print(f"💾 Folder Output Fitur  : {features_dir}")
print("=" * 65)

df_meta = pd.read_csv(csv_path)
print(f"✅ Metadata berhasil dibaca: {len(df_meta):,} citra terdaftar.")

## 2. Persiapan Custom Dataset & DataLoader PyTorch

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# Pipeline Preprocessing Standar ImageNet
transform_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class FashionDataset(Dataset):
    def __init__(self, metadata_df, img_dir, transform=None):
        self.df = metadata_df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['image_name']
        item_id = str(row['item_id'])
        img_path = os.path.join(self.img_dir, img_name)

        image = Image.open(img_path)
        if image.mode != 'RGB':
            image = image.convert('RGB')

        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)

        return image, img_name, item_id

# Inisialisasi Dataset & DataLoader (Batch Size 64)
dataset = FashionDataset(df_meta, images_dir, transform=transform_pipeline)
dataloader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"✅ DataLoader Siap: {len(dataset):,} gambar terbagi dalam {len(dataloader)} batch.")

## 3. Pemuatan & Modifikasi Arsitektur Model ResNet-50

In [ ]:
import torch.nn as nn
from torchvision import models

print("⏳ Mengunduh pretrained model ResNet-50 ImageNet...")
# Gunakan weights resmi terbaru dari PyTorch
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

# Modifikasi: Ganti Fully-Connected layer (fc) terakhir dengan Identity
# agar outputnya berupa vektor fitur 2048 dimensi (bukan prediksi 1000 kelas)
model.fc = nn.Identity()

# Pindahkan model ke GPU dan set ke mode Evaluasi (eval)
model = model.to(device)
model.eval()

print("=" * 65)
print("🧠 ARSITEKTUR MODEL BASELINE BERHASIL DISIAPKAN:")
print(" - Backbone Model       : ResNet-50 (Deep Residual Learning)")
print(" - Pretrained Weights   : ImageNet-1K (v2)")
print(" - Lapisan Output (fc)  : nn.Identity() (Feature Extractor Mode)")
print(" - Dimensi Output Vektor: 2048 Dimensi")
print(f" - Lokasi Komputasi     : {device}")
print("=" * 65)

## 4. Ekstraksi Fitur Berkelompok (Batch Feature Extraction) dengan GPU

In [ ]:
import numpy as np
import torch.nn.functional as F
from tqdm.auto import tqdm

print("🚀 Memulai ekstraksi fitur 3.608 gambar ke vektor 2048 dimensi...")

all_embeddings = []
all_image_names = []
all_item_ids = []

# Nonaktifkan perhitungan gradien untuk menghemat VRAM dan mempercepat inferensi
with torch.no_grad():
    for batch_images, batch_names, batch_items in tqdm(dataloader, desc="Ekstraksi Fitur ResNet50"):
        batch_images = batch_images.to(device)
        
        # 1. Forward pass melalui ResNet-50
        features = model(batch_images)  # Dimensi: [batch_size, 2048]
        
        # 2. Normalisasi L2 pada vektor fitur
        # Normalisasi L2 membuat panjang vektor = 1, sehingga Cosine Similarity = Dot Product!
        features = F.normalize(features, p=2, dim=1)
        
        # 3. Pindahkan hasil dari GPU ke CPU dalam format NumPy
        all_embeddings.append(features.cpu().numpy())
        all_image_names.extend(batch_names)
        all_item_ids.extend(batch_items)

# Gabungkan seluruh batch menjadi satu matriks 2D
embeddings_matrix = np.vstack(all_embeddings)  # Bentuk: [N, 2048]
image_names_array = np.array(all_image_names)
item_ids_array = np.array(all_item_ids)

print("\n🎉 EKSTRAKSI FITUR SELESAI!")
print("=" * 65)
print(f"📊 Bentuk Matriks Fitur (Embeddings) : {embeddings_matrix.shape}")
print(f"🖼️ Total Gambar Terekstraksi         : {len(image_names_array):,} file")
print(f"📏 Dimensi Vektor Tiap Gambar        : {embeddings_matrix.shape[1]} angka")
print(f"📐 Nilai Norm Rata-rata Vektor       : {np.linalg.norm(embeddings_matrix[0]):.4f} (L2-Normalized)")
print("=" * 65)

## 5. Simpan Vektor Fitur ke Google Drive (`CITRA-SERUPA/features/`)

In [ ]:
# Tentukan path file output
emb_file = os.path.join(features_dir, 'embeddings.npy')
names_file = os.path.join(features_dir, 'image_names.npy')
items_file = os.path.join(features_dir, 'item_ids.npy')

print("💾 Menyimpan fitur ke Google Drive...")
np.save(emb_file, embeddings_matrix)
np.save(names_file, image_names_array)
np.save(items_file, item_ids_array)

print("✅ Seluruh fitur visual berhasil disimpan secara permanen!")
print(f" - Matriks Embeddings : {emb_file} ({os.path.getsize(emb_file) / (1024**2):.2f} MB)")
print(f" - Daftar Nama File   : {names_file}")
print(f" - Daftar Item ID     : {items_file}")

## 6. Uji Coba Pencarian Rekomendasi Gambar Serupa (Top-5 Retrieval Demo)

In [ ]:
import matplotlib.pyplot as plt
import random

# 1. Pilih 1 indeks gambar secara acak sebagai KUERI (Query)
query_idx = random.randint(0, len(embeddings_matrix) - 1)
query_name = image_names_array[query_idx]
query_item = item_ids_array[query_idx]
query_vec = embeddings_matrix[query_idx]

# 2. Hitung Cosine Similarity terhadap seluruh 3.608 gambar di galeri
# Karena vektor sudah L2-normalized, Cosine Similarity = Perkalian Dot Product!
similarity_scores = np.dot(embeddings_matrix, query_vec)

# 3. Ambil indeks dengan skor kemiripan tertinggi (Top 6: nomor 1 adalah dirinya sendiri)
top_indices = np.argsort(similarity_scores)[::-1][:6]

# 4. Visualisasikan Hasil Rekomendasi
fig, axes = plt.subplots(1, 6, figsize=(18, 4))

# Tampilkan Gambar Kueri
query_img_path = os.path.join(images_dir, query_name)
axes[0].imshow(Image.open(query_img_path))
axes[0].set_title(f"🎯 KUERI (Input)\n{query_item}\nPose: {query_name.split('_')[-1].replace('.jpg','')}", fontsize=10, color='blue', fontweight='bold')
axes[0].axis('off')

# Tampilkan 5 Gambar Rekomendasi Teratas
for rank, idx in enumerate(top_indices[1:], 1):
    rec_name = image_names_array[idx]
    rec_item = item_ids_array[idx]
    rec_score = similarity_scores[idx] * 100
    is_match = (rec_item == query_item)  # Ground Truth match
    
    rec_img_path = os.path.join(images_dir, rec_name)
    axes[rank].imshow(Image.open(rec_img_path))
    match_label = "SAMA (Match) ✅" if is_match else "BEDA (Similar)"
    title_color = 'green' if is_match else 'black'
    
    axes[rank].set_title(
        f"Rank #{rank} ({rec_score:.1f}%)\n{rec_item}\n{match_label}",
        fontsize=9,
        color=title_color
    )
    axes[rank].axis('off')

plt.tight_layout()
plt.show()

print("=" * 65)
print(f"🎯 Gambar Kueri : {query_name} (Produk: {query_item})")
print("🔍 Rekomendasi 5 Gambar Teratas:")
for rank, idx in enumerate(top_indices[1:], 1):
    is_match = (item_ids_array[idx] == query_item)
    print(f"  Rank #{rank}: {image_names_array[idx]} | Skor: {similarity_scores[idx]*100:.2f}% | Produk Sama: {'YA ✅' if is_match else 'TIDAK'}")
print("=" * 65)

## 7. Kesimpulan & Status Model Baseline

In [ ]:
print("=" * 65)
print("🏆 KESIMPULAN MODEL BASELINE EKSTRAKSI FITUR RESNET-50")
print("=" * 65)
print(f"1. Model Arsitektur  : ResNet-50 Pretrained ImageNet")
print(f"2. Dimensi Fitur     : 2048 Dimensi (L2-Normalized)")
print(f"3. Total Citra Basis : {len(embeddings_matrix):,} Gambar")
print(f"4. Metrik Kemiripan  : Cosine Similarity (Dot Product)")
print(f"5. Lokasi Fitur Drive: {features_dir}")
print("-" * 65)
print("🎉 SISTEM EKSTRAKSI FITUR BASELINE 100% SUKSES DAN SIAP DIEVALUASI!")
print("=" * 65)